# 💳 Credit Risk Prediction — Loan Default Analysis

**Autor:** Rafael Reghine Munhoz  
**Dataset:** Give Me Some Credit — Kaggle  
**Objetivo:** Prever a probabilidade de inadimplência utilizando Machine Learning.

---
### ▶️ Como usar:
1. Baixe o dataset em: https://www.kaggle.com/competitions/GiveMeSomeCredit/data
2. Execute a **Célula 1** para instalar apenas o que falta
3. **Runtime → Restart Runtime**
4. Execute a **Célula 2** para upload do `cs-training.csv`
5. Execute todas as células em ordem
---

## ⚙️ Célula 1 — Instalar apenas dependências ausentes

In [ ]:
# Instala apenas o que o Colab não tem nativamente
# Usa as versões já instaladas de numpy, pandas, sklearn, matplotlib, seaborn
!pip install imbalanced-learn -q
!pip install shap -q

import numpy as np
import sklearn
import xgboost

print(f'✅ NumPy:     {np.__version__}')
print(f'✅ Sklearn:   {sklearn.__version__}')
print(f'✅ XGBoost:   {xgboost.__version__}')
print()
print('⚠️  Vá em Runtime → Restart Runtime')
print('   Depois execute a partir da Célula 2')

## 📁 Célula 2 — Upload do Dataset

In [ ]:
from google.colab import files
import io

print('📂 Selecione o arquivo cs-training.csv:')
uploaded = files.upload()

for fname in uploaded.keys():
    print(f'✅ Carregado: {fname} ({len(uploaded[fname])/1024/1024:.1f} MB)')

## 1. Importação de Bibliotecas

In [ ]:
import pandas as pd
import numpy as np
import io
import pickle
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)

from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
import shap

# Estilo dos gráficos
try:
    plt.style.use('seaborn-v0_8-whitegrid')
except:
    plt.style.use('seaborn-whitegrid')

pd.set_option('display.float_format', '{:.2f}'.format)

print(f'✅ NumPy:      {np.__version__}')
print(f'✅ Pandas:     {pd.__version__}')
print('✅ Todas as bibliotecas importadas com sucesso!')

## 2. Carregamento e Inspeção dos Dados

In [ ]:
# Carregar dataset
fname = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[fname]), index_col=0)

# Renomear colunas para português
df.columns = [
    'inadimplente',
    'utilizacao_credito_rotativo',
    'idade',
    'atrasos_30_59_dias',
    'taxa_endividamento',
    'renda_mensal',
    'qtd_linhas_credito',
    'atrasos_90_dias',
    'qtd_emprestimos_imoveis',
    'atrasos_60_89_dias',
    'qtd_dependentes'
]

print(f'✅ Dataset carregado: {df.shape[0]:,} linhas x {df.shape[1]} colunas')
print()

contagem = df['inadimplente'].value_counts()
print('📊 Distribuição do target:')
print(f'  Adimplentes (0):   {contagem[0]:,} ({contagem[0]/len(df):.1%})')
print(f'  Inadimplentes (1): {contagem[1]:,} ({contagem[1]/len(df):.1%})')
print(f'\n⚠️  Dataset desbalanceado — usaremos SMOTE para corrigir.')
df.head()

In [ ]:
# Estatísticas descritivas
df.describe().round(2)

In [ ]:
# Valores nulos
nulos = df.isnull().sum()
pct   = (nulos / len(df) * 100).round(2)
print('🔍 Colunas com valores nulos:')
print(pd.DataFrame({'Nulos': nulos, '% do Total': pct}).query('Nulos > 0').to_string())

## 3. Análise Exploratória (EDA)

In [ ]:
# Distribuição do target
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
cores  = ['#2ecc71', '#e74c3c']
labels = ['Adimplente', 'Inadimplente']
counts = df['inadimplente'].value_counts().sort_index()

bars = axes[0].bar(labels, counts.values, color=cores, edgecolor='white', linewidth=1.5)
axes[0].set_title('Adimplente vs Inadimplente', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Quantidade')
for bar, val in zip(bars, counts.values):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 300, f'{val:,}',
                 ha='center', fontweight='bold', fontsize=11)

axes[1].pie(counts.values, labels=labels, colors=cores,
            autopct='%1.1f%%', startangle=90, textprops={'fontsize': 11})
axes[1].set_title('Proporção', fontsize=13, fontweight='bold')

plt.suptitle('Desbalanceamento da Classe Target', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('distribuicao_target.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Distribuição das variáveis (sem outliers extremos)
features_viz = [
    'idade', 'renda_mensal', 'taxa_endividamento',
    'utilizacao_credito_rotativo', 'qtd_linhas_credito', 'qtd_dependentes'
]

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

for i, col in enumerate(features_viz):
    data = df[col].dropna()
    data = data[data <= data.quantile(0.99)]
    axes[i].hist(data, bins=40, color='#3498db', edgecolor='white', alpha=0.85)
    axes[i].set_title(col.replace('_', ' ').title(), fontsize=11, fontweight='bold')
    axes[i].set_ylabel('Frequência')

plt.suptitle('Distribuição das Variáveis (p99)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('distribuicao_variaveis.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Taxa de inadimplência por faixa etária
df['faixa_etaria'] = pd.cut(
    df['idade'],
    bins=[0, 25, 35, 45, 55, 65, 120],
    labels=['< 25', '25-35', '35-45', '45-55', '55-65', '65+']
)

taxa = df.groupby('faixa_etaria', observed=True)['inadimplente'].mean() * 100

plt.figure(figsize=(10, 5))
bars = plt.bar(taxa.index, taxa.values, color='#e74c3c', alpha=0.8, edgecolor='white')
plt.title('Taxa de Inadimplência por Faixa Etária', fontsize=14, fontweight='bold')
plt.xlabel('Faixa Etária')
plt.ylabel('Taxa de Inadimplência (%)')
for bar, val in zip(bars, taxa.values):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height() + 0.05, f'{val:.1f}%',
             ha='center', fontweight='bold', fontsize=10)
plt.tight_layout()
plt.savefig('inadimplencia_por_idade.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Impacto do histórico de atrasos
atrasos_cols  = ['atrasos_30_59_dias', 'atrasos_60_89_dias', 'atrasos_90_dias']
titulos_atr   = ['Atrasos 30-59 dias', 'Atrasos 60-89 dias', 'Atrasos 90+ dias']
cores_atr     = ['#f39c12', '#e67e22', '#e74c3c']

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, col, titulo, cor in zip(axes, atrasos_cols, titulos_atr, cores_atr):
    t = df.groupby(df[col].clip(0, 5))['inadimplente'].mean() * 100
    bars = ax.bar(t.index, t.values, color=cor, alpha=0.85, edgecolor='white')
    ax.set_title(titulo, fontsize=12, fontweight='bold')
    ax.set_xlabel('Qtd. de Atrasos')
    ax.set_ylabel('Taxa de Inadimplência (%)')
    for bar, val in zip(bars, t.values):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.3, f'{val:.0f}%',
                ha='center', fontweight='bold', fontsize=9)

plt.suptitle('Histórico de Atrasos vs Inadimplência', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('atrasos_inadimplencia.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Matriz de correlação
plt.figure(figsize=(12, 8))
corr = df.drop('faixa_etaria', axis=1).corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0, square=True,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
plt.title('Matriz de Correlação', fontsize=14, fontweight='bold')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('correlacao.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Pré-processamento

In [ ]:
df = df.drop('faixa_etaria', axis=1)

X = df.drop('inadimplente', axis=1).copy()
y = df['inadimplente']

# Tratar outliers extremos (clip p99)
for col in ['renda_mensal', 'taxa_endividamento', 'utilizacao_credito_rotativo']:
    p99 = X[col].quantile(0.99)
    X[col] = X[col].clip(upper=p99)
    print(f'  {col}: p99 = {p99:.2f}')

# Imputar nulos com mediana
imputer  = SimpleImputer(strategy='median')
X_imp    = pd.DataFrame(imputer.fit_transform(X), columns=X.columns)

print(f'\n✅ Nulos após imputação: {X_imp.isnull().sum().sum()}')
print(f'✅ Shape: {X_imp.shape}')

In [ ]:
# Split treino / teste
X_train, X_test, y_train, y_test = train_test_split(
    X_imp, y, test_size=0.2, random_state=42, stratify=y
)
print(f'Treino: {X_train.shape[0]:,} | Teste: {X_test.shape[0]:,}')

In [ ]:
# SMOTE — balancear classes
print('⚖️  Aplicando SMOTE...')
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

bal = pd.Series(y_train_bal).value_counts()
print(f'✅ Adimplentes:   {bal[0]:,}')
print(f'✅ Inadimplentes: {bal[1]:,}')

In [ ]:
# Normalização
scaler          = StandardScaler()
X_train_scaled  = scaler.fit_transform(X_train_bal)
X_test_scaled   = scaler.transform(X_test)
print('✅ Normalização concluída')

## 5. Modelagem

In [ ]:
modelos = {
    'Regressão Logística': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(
        n_estimators=200, max_depth=10,
        min_samples_leaf=5, random_state=42, n_jobs=-1
    ),
    'XGBoost': XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        eval_metric='logloss', random_state=42
    )
}

resultados = {}
print('🏋️  Treinando modelos...\n')

for nome, modelo in modelos.items():
    print(f'  → {nome}...')
    modelo.fit(X_train_scaled, y_train_bal)
    y_pred  = modelo.predict(X_test_scaled)
    y_proba = modelo.predict_proba(X_test_scaled)[:, 1]
    auc     = roc_auc_score(y_test, y_proba)
    resultados[nome] = {'modelo': modelo, 'y_pred': y_pred,
                        'y_proba': y_proba, 'auc': auc}
    print(f'     AUC-ROC: {auc:.4f}')

melhor = max(resultados, key=lambda k: resultados[k]['auc'])
print(f'\n🏆 Melhor modelo: {melhor} (AUC = {resultados[melhor]["auc"]:.4f})')

In [ ]:
# Relatório detalhado
for nome, res in resultados.items():
    print(f'\n{"="*50}')
    print(f'📋 {nome} — AUC: {res["auc"]:.4f}')
    print('='*50)
    print(classification_report(
        y_test, res['y_pred'],
        target_names=['Adimplente', 'Inadimplente']
    ))

In [ ]:
# Curva ROC
plt.figure(figsize=(9, 6))
cores_roc = ['#95a5a6', '#2ecc71', '#e74c3c']

for (nome, res), cor in zip(resultados.items(), cores_roc):
    fpr, tpr, _ = roc_curve(y_test, res['y_proba'])
    lw = 3 if nome == melhor else 1.8
    ls = '-' if nome == melhor else '--'
    plt.plot(fpr, tpr, label=f"{nome} (AUC={res['auc']:.4f})",
             color=cor, lw=lw, linestyle=ls)

plt.plot([0,1],[0,1],'k--', lw=1.2, alpha=0.5, label='Aleatório')
plt.xlabel('Taxa de Falso Positivo', fontsize=12)
plt.ylabel('Taxa de Verdadeiro Positivo', fontsize=12)
plt.title('Curva ROC — Comparativo de Modelos', fontsize=14, fontweight='bold')
plt.legend(loc='lower right', fontsize=10)
plt.tight_layout()
plt.savefig('roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Matrizes de confusão
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, (nome, res) in zip(axes, resultados.items()):
    cm = confusion_matrix(y_test, res['y_pred'])
    ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=['Adimplente', 'Inadimplente']
    ).plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(f'{nome}\nAUC={res["auc"]:.4f}', fontsize=11, fontweight='bold')

plt.suptitle('Matrizes de Confusão', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Feature Importance — Random Forest
rf = resultados['Random Forest']['modelo']
fi = pd.DataFrame({'Feature': X.columns,
                   'Importância': rf.feature_importances_
                  }).sort_values('Importância', ascending=True)

cores_fi = ['#e74c3c' if i >= len(fi)-3 else '#3498db' for i in range(len(fi))]

plt.figure(figsize=(10, 6))
plt.barh(fi['Feature'], fi['Importância'], color=cores_fi, edgecolor='white')
plt.xlabel('Importância Relativa', fontsize=12)
plt.title('Feature Importance — Random Forest', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Explicabilidade com SHAP

In [ ]:
print('🔍 Calculando SHAP values (~1 minuto)...')
xgb = resultados['XGBoost']['modelo']

# Amostra para agilizar
np.random.seed(42)
idx = np.random.choice(len(X_test_scaled), size=2000, replace=False)
X_shap = X_test_scaled[idx]

explainer   = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_shap)
print('✅ SHAP calculado!')

In [ ]:
# Summary Plot
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, X_shap,
                  feature_names=X.columns.tolist(),
                  show=False, plot_size=None)
plt.title('SHAP — Impacto das Variáveis (XGBoost)',
          fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# SHAP Bar — importância média absoluta
shap_df = pd.DataFrame({
    'Feature': X.columns,
    'SHAP_mean': np.abs(shap_values).mean(axis=0)
}).sort_values('SHAP_mean', ascending=True)

cores_sb = ['#e74c3c' if i >= len(shap_df)-3 else '#3498db'
            for i in range(len(shap_df))]

plt.figure(figsize=(10, 6))
plt.barh(shap_df['Feature'], shap_df['SHAP_mean'],
         color=cores_sb, edgecolor='white')
plt.xlabel('Impacto médio absoluto (SHAP)', fontsize=12)
plt.title('Importância via SHAP — XGBoost', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Insights de Negócio

In [ ]:
print('📊 Resumo comparativo:')
for nome, res in resultados.items():
    flag = ' 🏆' if nome == melhor else ''
    print(f'  {nome}: AUC = {res["auc"]:.4f}{flag}')

print("""
╔══════════════════════════════════════════════════════╗
║           PRINCIPAIS INSIGHTS DE NEGÓCIO             ║
╠══════════════════════════════════════════════════════╣
║  1. MAIOR PREDITOR: atrasos_90_dias                  ║
║     2+ atrasos graves = risco muito alto             ║
║                                                      ║
║  2. ALERTA: utilização crédito rotativo > 80%        ║
║     Indica stress financeiro                         ║
║                                                      ║
║  3. PERFIL DE RISCO ALTO:                            ║
║     Jovens + alta utilização + histórico de atrasos  ║
║                                                      ║
║  4. RECOMENDAÇÕES:                                   ║
║     • Alerta: atrasos_90_dias >= 2                   ║
║     • Revisar política: utilização > 80%             ║
║     • Score por faixa etária                         ║
╚══════════════════════════════════════════════════════╝
""")

## 8. Salvar e Baixar Modelos

In [ ]:
import os
os.makedirs('models', exist_ok=True)

with open('models/best_model.pkl', 'wb') as f:
    pickle.dump(resultados['XGBoost']['modelo'], f)
with open('models/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)
with open('models/imputer.pkl', 'wb') as f:
    pickle.dump(imputer, f)

print(f'✅ Modelos salvos! AUC final (XGBoost): {resultados["XGBoost"]["auc"]:.4f}')

# Download
from google.colab import files as colab_files
colab_files.download('models/best_model.pkl')
colab_files.download('models/scaler.pkl')
colab_files.download('models/imputer.pkl')
print('📥 Download concluído! Coloque os .pkl na pasta /models do projeto.')